# DeepConvLSTM Replication Notebook (Safe Pipeline + Simplified PTQ/QAT)

This notebook keeps the **methodologically safe** v1 pipeline and simplifies PTQ/QAT export/evaluation using the direct TFLite conversion style from `replication_deepconvlstm_v2.ipynb`.

## Scope
- Preserve v1 safeguards: train-only normalization, per-user-safe windowing, and train/val/test split.
- Keep both split protocols: `random_stratified` and `user_holdout`.
- Simplify PTQ and QAT to one shared int8 conversion/evaluation pattern.
- Use one calibration variant (`authorcal`): representative samples from `X_test`.
- Keep reproducibility artifacts (including split hash checks).

This notebook reuses `src/` modules directly to avoid logic drift.

In [1]:
from pathlib import Path
import sys
import os
import json
import copy
import importlib.util

# Toggle this before running the cell if your kernel crashes on GPU.
USE_GPU = True
if not USE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# --- TensorFlow GPU/XLA runtime guardrails (must run before importing project modules) ---
# In this conda env, NVIDIA CUDA libs and ptxas may live under site-packages/nvidia/*.
py_major, py_minor = sys.version_info[:2]
nvidia_root = Path(sys.prefix) / f"lib/python{py_major}.{py_minor}/site-packages/nvidia"
cuda_nvcc_root = nvidia_root / "cuda_nvcc"
ptxas_bin_dir = cuda_nvcc_root / "bin"
ptxas_path = ptxas_bin_dir / "ptxas"

if USE_GPU and nvidia_root.exists():
    # Ensure CUDA shared libraries (including libnvrtc) are discoverable.
    lib_dirs = [p for p in sorted(nvidia_root.glob("*/lib")) if p.is_dir()]
    existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
    ld_entries = existing_ld.split(":") if existing_ld else []
    for lib_dir in reversed(lib_dirs):
        s = str(lib_dir)
        if s not in ld_entries:
            ld_entries.insert(0, s)
    os.environ["LD_LIBRARY_PATH"] = ":".join(ld_entries)

    # Ensure ptxas is on PATH for XLA CUDA compilation.
    if ptxas_path.exists():
        path_entries = os.environ.get("PATH", "").split(":") if os.environ.get("PATH") else []
        if str(ptxas_bin_dir) not in path_entries:
            os.environ["PATH"] = f"{ptxas_bin_dir}:{os.environ.get('PATH', '')}".strip(":")

        # Hint XLA where CUDA toolchain data lives.
        xla_flag = f"--xla_gpu_cuda_data_dir={cuda_nvcc_root}"
        existing_xla = os.environ.get("XLA_FLAGS", "")
        if xla_flag not in existing_xla:
            os.environ["XLA_FLAGS"] = (existing_xla + " " + xla_flag).strip()

required_modules = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "IPython": "ipython",
    "yaml": "PyYAML",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "tensorflow": "tensorflow",
    "tensorflow_model_optimization": "tensorflow-model-optimization",
}
missing = [f"{mod} (pip package: {pkg})" for mod, pkg in required_modules.items() if importlib.util.find_spec(mod) is None]
if missing:
    missing_text = "\n- ".join(missing)
    raise ModuleNotFoundError(
        "Missing required notebook dependencies:\n- "
        + missing_text
        + "\n\nActivate tinymlproj and install dependencies:\n"
        + "conda activate tinymlproj && conda env update -n tinymlproj -f environment.yml --prune"
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import accuracy_score, f1_score

import tensorflow as tf

# Configure TF GPU memory growth early (before any real TF GPU allocations happen)
if USE_GPU:
    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception as e:
            print(f"[WARN] Could not set memory growth for {g}: {e}")

print("TF version:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# Ensure repo root is importable when launched from different working directories.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import load_yaml
from src.utils.runtime import check_tensorflow_runtime
from src.utils.repro import load_json, set_global_seed, dump_json
from src.utils.artifacts import baseline_ckpt_path, history_path, split_npz_path

from src.data.load_wisdm import load_wisdm_dataframe
from src.data.preprocess_zhou2025 import preprocess_zhou2025
from src.data.build_dataset import build_dataset_for_protocol
from src.data.io import load_split_arrays

from src.train.train_baseline import train_baseline_for_protocol
from src.eval.eval_baseline import evaluate_baseline_for_protocol

nvrtc_candidates = []
if nvidia_root.exists():
    for lib_dir in sorted(nvidia_root.glob("*/lib")):
        nvrtc_candidates.extend(sorted(lib_dir.glob("libnvrtc.so*")))

if USE_GPU and not nvrtc_candidates:
    raise RuntimeError(
        "GPU mode requested but libnvrtc.so was not found in tinymlproj. "
        "Install/refresh env with: conda activate tinymlproj && "
        "conda env update -n tinymlproj -f environment.yml --prune"
    )

print(f"Repo root: {REPO_ROOT}")
print(f"USE_GPU: {USE_GPU}")
print(f"nvidia_root exists: {nvidia_root.exists()} at {nvidia_root}")
print(f"ptxas found: {ptxas_path.exists()} at {ptxas_path if ptxas_path.exists() else 'N/A'}")
print(f"libnvrtc candidates: {[str(p) for p in nvrtc_candidates[:3]]}")
print(f"XLA_FLAGS: {os.environ.get('XLA_FLAGS', '')}")

2026-03-02 12:32:53.282729: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-02 12:32:53.282847: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-02 12:32:53.284110: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-02 12:32:53.402896: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.14.1
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Repo root: /home/dellio/github/har-mcu
USE_GPU: True
nvidia_root exists: True at /home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/nvidia
ptxas found: True at /home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/nvidia/cuda_nvcc/bin/ptxas
libnvrtc candidates: ['/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.11.2']
XLA_FLAGS: --xla_gpu_cuda_data_dir=/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/nvidia/cuda_nvcc


2026-03-02 12:32:55.659120: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 12:32:55.680791: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 12:32:55.680857: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.


In [2]:
CONFIG_PATH = REPO_ROOT / "configs/default.yaml"
cfg_default = load_yaml(CONFIG_PATH)

if not USE_GPU:
    cfg_default = copy.deepcopy(cfg_default)
    cfg_default.setdefault("env", {})["require_gpu"] = False

runtime_status = check_tensorflow_runtime(cfg_default)
display(pd.DataFrame([runtime_status]))

if "error" in runtime_status:
    raise RuntimeError(runtime_status["error"])

print("Loaded config:", CONFIG_PATH)



,tensorflow_ok,version_ok,gpu_ok,tensorflow_version,gpus
0,True,True,True,2.14.1,[/physical_device:GPU:0]


Loaded config: /home/dellio/github/har-mcu/configs/default.yaml


In [3]:
# Parameters
RUN_MODE = "full"  # "quick" or "full"
WINDOW_SIZE = 100
PROTOCOLS = ["random_stratified", "user_holdout"]
RUN_QAT = True
FORCE_RETRAIN = False
AUTHOR_STYLE_REP_SAMPLES = 100
FAIL_FAST = False

print({
    "RUN_MODE": RUN_MODE,
    "WINDOW_SIZE": WINDOW_SIZE,
    "PROTOCOLS": PROTOCOLS,
    "RUN_QAT": RUN_QAT,
    "FORCE_RETRAIN": FORCE_RETRAIN,
    "AUTHOR_STYLE_REP_SAMPLES": AUTHOR_STYLE_REP_SAMPLES,
    "FAIL_FAST": FAIL_FAST,
})

{'RUN_MODE': 'full', 'WINDOW_SIZE': 100, 'PROTOCOLS': ['random_stratified', 'user_holdout'], 'RUN_QAT': True, 'FORCE_RETRAIN': False, 'AUTHOR_STYLE_REP_SAMPLES': 100, 'FAIL_FAST': False}


In [4]:
def build_notebook_cfg(base_cfg, run_mode, window_size, protocols):
    cfg = copy.deepcopy(base_cfg)
    cfg["window_size_default"] = int(window_size)
    cfg["split_protocols"] = list(protocols)

    if run_mode not in {"quick", "full"}:
        raise ValueError("RUN_MODE must be 'quick' or 'full'")

    quant_cfg = cfg.setdefault("quant", {})
    ptq_cfg = quant_cfg.setdefault("ptq", {})
    qat_cfg = quant_cfg.setdefault("qat", {})

    # Notebook PTQ/QAT uses v2-style representative samples from test split.
    ptq_cfg["representative_samples"] = int(AUTHOR_STYLE_REP_SAMPLES)
    qat_cfg["representative_samples"] = int(AUTHOR_STYLE_REP_SAMPLES)
    qat_cfg["enabled"] = bool(qat_cfg.get("enabled", True))

    if run_mode == "quick":
        cfg.setdefault("smoke", {})["enabled"] = True
        cfg["train"]["epochs"] = 1
        cfg["smoke"]["max_windows_per_class"] = 200
        cfg["quant"]["qat"]["epochs"] = 1
    else:
        cfg.setdefault("smoke", {})["enabled"] = False
        cfg["smoke"]["max_windows_per_class"] = None

    return cfg

cfg_effective = build_notebook_cfg(
    cfg_default,
    RUN_MODE,
    WINDOW_SIZE,
    PROTOCOLS,
    )
set_global_seed(int(cfg_effective["seed"]))

CALIBRATION_VARIANT = "authorcal"
EXPECTED_LABEL_POLICY = "drop_cross_boundary"

preview = {
    "window_size_default": cfg_effective["window_size_default"],
    "split_protocols": cfg_effective["split_protocols"],
    "label_policy": cfg_effective.get("label_policy"),
    "epochs": cfg_effective["train"]["epochs"],
    "ptq_representative_samples": cfg_effective["quant"]["ptq"]["representative_samples"],
    "qat_representative_samples": cfg_effective["quant"]["qat"]["representative_samples"],
    "calibration_variant": CALIBRATION_VARIANT,
    "smoke_enabled": cfg_effective["smoke"]["enabled"],
    "smoke_max_windows_per_class": cfg_effective["smoke"].get("max_windows_per_class"),
}
print(json.dumps(preview, indent=2))

{
  "window_size_default": 100,
  "split_protocols": [
    "random_stratified",
    "user_holdout"
  ],
  "label_policy": "drop_cross_boundary",
  "epochs": 50,
  "ptq_representative_samples": 100,
  "qat_representative_samples": 100,
  "calibration_variant": "authorcal",
  "smoke_enabled": false,
  "smoke_max_windows_per_class": null
}


In [5]:
raw_df, sanity = load_wisdm_dataframe(cfg_effective)
clean_df, pre_stats = preprocess_zhou2025(raw_df, cfg_effective)

class_counts = raw_df["activity"].value_counts().rename("count").reset_index().rename(columns={"index": "activity"})

sanity_table = pd.DataFrame([
    {
        "rows_raw": int(len(raw_df)),
        "rows_after_preprocess": int(len(clean_df)),
        "missing_values": int(sanity["missing_values"]),
        "zero_timestamps": int(sanity["zero_timestamps"]),
        "unique_users": int(raw_df["user"].nunique()),
    }
])

display(sanity_table)
display(class_counts)

paper_readme_rows = 1098207
local_rows = int(len(raw_df))
if local_rows != paper_readme_rows:
    print(
        f"Note: local CSV rows ({local_rows}) differ from README/paper-stated rows ({paper_readme_rows}). "
        "Replication should be interpreted directionally."
    )


,rows_raw,rows_after_preprocess,missing_values,zero_timestamps,unique_users
0,1073623,1073623,0,0,36


,activity,count
0,Walking,417901
1,Jogging,324600
2,Upstairs,122598
3,Downstairs,100192
4,Sitting,59939
5,Standing,48393


Note: local CSV rows (1073623) differ from README/paper-stated rows (1098207). Replication should be interpreted directionally.


In [7]:
dataset_cards = {}
split_rows = []
class_rows = []

for protocol in PROTOCOLS:
    out = build_dataset_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    card = load_json(out["artifacts"]["datacard"])
    dataset_cards[protocol] = card

    split_rows.append(
        {
            "protocol": protocol,
            "window_size": card["windowing"]["window_size"],
            "step": card["windowing"]["step"],
            "label_policy": card["windowing"].get("label_policy"),
            "candidate_windows": card["windowing"]["candidate_windows"],
            "windows_final": card["windowing"]["windows_final"],
            "train_size": card["split"]["train_size"],
            "val_size": card["split"]["val_size"],
            "test_size": card["split"]["test_size"],
            "split_hash": card["split"]["split_hash"],
        }
    )

    for cls, counts in card["counts"].items():
        class_rows.append(
            {
                "protocol": protocol,
                "activity": cls,
                "train": counts["train"],
                "val": counts["val"],
                "test": counts["test"],
            }
        )

split_df = pd.DataFrame(split_rows)
class_df = pd.DataFrame(class_rows)

# Methodology guardrails (must remain leakage-safe unlike v2 pitfalls).
if cfg_effective.get("label_policy") != EXPECTED_LABEL_POLICY:
    raise RuntimeError(
        f"Config label_policy is {cfg_effective.get('label_policy')}, expected {EXPECTED_LABEL_POLICY}"
    )

for protocol, card in dataset_cards.items():
    if card["split"]["val_size"] <= 0:
        raise RuntimeError(f"{protocol}: validation split missing")
    if card["split"]["test_size"] <= 0:
        raise RuntimeError(f"{protocol}: test split missing")

    # Optional datacard consistency check: some historical cards may omit this field.
    lp_card = card["windowing"].get("label_policy")
    if lp_card is not None and lp_card != EXPECTED_LABEL_POLICY:
        raise RuntimeError(
            f"{protocol}: datacard label_policy is {lp_card}, expected {EXPECTED_LABEL_POLICY}"
        )

    if not card["split"].get("split_hash"):
        raise RuntimeError(f"{protocol}: split_hash missing")

display(split_df)
display(class_df)

,protocol,window_size,step,label_policy,candidate_windows,windows_final,train_size,val_size,test_size,split_hash
0,random_stratified,100,50,None,21421,20709,11100,2775,6834,e8b3eefc294da8de
1,user_holdout,100,50,None,21421,20709,10945,2713,7051,8f23a2e13f210a67


,protocol,activity,train,val,test
0,random_stratified,Downstairs,954,238,587
1,random_stratified,Jogging,3425,856,2109
2,random_stratified,Sitting,619,155,381
3,random_stratified,Standing,493,123,304
4,random_stratified,Upstairs,1186,297,730
5,random_stratified,Walking,4423,1106,2723
6,user_holdout,Downstairs,896,250,633
7,user_holdout,Jogging,3159,939,2292
8,user_holdout,Sitting,816,70,269
9,user_holdout,Standing,606,46,268


In [8]:
baseline_train_rows = []

for protocol in PROTOCOLS:
    ckpt_path = baseline_ckpt_path(cfg_effective["paths"]["checkpoints_dir"], WINDOW_SIZE, protocol)
    hist_path = history_path(cfg_effective["paths"]["checkpoints_dir"], WINDOW_SIZE, protocol)

    if ckpt_path.exists() and not FORCE_RETRAIN:
        train_out = {
            "checkpoint": str(ckpt_path),
            "history": str(hist_path) if hist_path.exists() else None,
            "epochs_ran": 0,
            "final_val_accuracy": None,
            "reused_checkpoint": True,
        }
    else:
        train_out = train_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
        train_out["reused_checkpoint"] = False

    train_out["protocol"] = protocol
    baseline_train_rows.append(train_out)

baseline_train_df = pd.DataFrame(baseline_train_rows)
display(baseline_train_df)


,checkpoint,history,epochs_ran,final_val_accuracy,reused_checkpoint,protocol
0,/home/dellio/github/har-mcu/checkpoints/deepco...,/home/dellio/github/har-mcu/checkpoints/histor...,0,None,True,random_stratified
1,/home/dellio/github/har-mcu/checkpoints/deepco...,/home/dellio/github/har-mcu/checkpoints/histor...,0,None,True,user_holdout


In [9]:
baseline_eval_metrics = {}
baseline_eval_rows = []

for protocol in PROTOCOLS:
    out = evaluate_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    m = load_json(out["metrics_json"])
    baseline_eval_metrics[protocol] = m

    baseline_eval_rows.append(
        {
            "protocol": protocol,
            "accuracy": m["accuracy"],
            "macro_f1": m["macro_f1"],
            "confusion_plot": m["confusion_plot"],
            "metrics_json": out["metrics_json"],
            "report_md": out["report_md"],
        }
    )

baseline_eval_df = pd.DataFrame(baseline_eval_rows)
display(baseline_eval_df)


2026-03-02 12:34:19.013419: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 12:34:19.015136: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 12:34:19.015193: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 12:34:19.231298: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-02 12:34:19.231943: I tensorflow/compile

,protocol,accuracy,macro_f1,confusion_plot,metrics_json,report_md
0,random_stratified,0.511853,0.454694,/home/dellio/github/har-mcu/reports/confusion_...,/home/dellio/github/har-mcu/reports/baseline_T...,/home/dellio/github/har-mcu/reports/baseline_T...
1,user_holdout,0.070770,0.062999,/home/dellio/github/har-mcu/reports/confusion_...,/home/dellio/github/har-mcu/reports/baseline_T...,/home/dellio/github/har-mcu/reports/baseline_T...


In [10]:
ptq_export_metrics = {}
ptq_eval_metrics = {}
ptq_rows = []

def evaluate_int8_tflite_model(model_path, X_test, y_test):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    input_det = interpreter.get_input_details()[0]
    output_det = interpreter.get_output_details()[0]

    in_scale, in_zero = input_det["quantization"]
    in_dtype = input_det["dtype"]

    if in_scale == 0:
        raise RuntimeError("Input tensor is not quantized; expected int8/uint8 model input")

    dtype_info = np.iinfo(in_dtype)
    y_pred = []
    y_true = []

    for i in range(len(X_test)):
        x = X_test[i:i+1]
        x_q = np.round(x / in_scale + in_zero)
        x_q = np.clip(x_q, dtype_info.min, dtype_info.max).astype(in_dtype)

        interpreter.set_tensor(input_det["index"], x_q)
        interpreter.invoke()
        out = interpreter.get_tensor(output_det["index"])

        y_pred.append(int(np.argmax(out, axis=-1)[0]))
        y_true.append(int(y_test[i]))

    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))
    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "model_size_kb": float(Path(model_path).stat().st_size / 1024.0),
        "input_dtype": str(input_det["dtype"]),
        "output_dtype": str(output_det["dtype"]),
    }

def convert_to_int8_tflite(model, X_test, out_path, rep_samples):
    model.input.set_shape((1,) + tuple(model.input.shape[1:]))
    rep_count = int(min(rep_samples, len(X_test)))

    def representative_dataset_gen():
        for sample in tf.data.Dataset.from_tensor_slices(X_test).batch(1).take(rep_count):
            yield [tf.cast(sample, tf.float32)]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(converter.convert())
    return rep_count

def write_quant_report_md(path, title, export_json):
    with path.open("w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")
        f.write(f"- status: `{export_json['status']}`\n")
        f.write(f"- representative_source: `test`\n")
        f.write(f"- representative_samples: `{export_json['representative_samples']}`\n")
        if export_json.get("tflite_model"):
            f.write(f"- tflite_model: `{export_json['tflite_model']}`\n")
        if export_json.get("model_size_kb") is not None:
            f.write(f"- model_size_kb: {export_json['model_size_kb']:.2f}\n")
        if export_json.get("error"):
            f.write(f"- error: `{export_json['error']}`\n")

for protocol in PROTOCOLS:
    arrays = load_split_arrays(cfg_effective["paths"]["processed_dir"], WINDOW_SIZE, protocol)
    X_test = arrays["X_test"]
    y_test = arrays["y_test"]

    ckpt_path = baseline_ckpt_path(cfg_effective["paths"]["checkpoints_dir"], WINDOW_SIZE, protocol)
    model = tf.keras.models.load_model(ckpt_path)

    tflite_path = Path(cfg_effective["paths"]["models_tflite_dir"]) / (
        f"deepconv_lstm_T{WINDOW_SIZE}_P{protocol}_ptq_int8_{CALIBRATION_VARIANT}.tflite"
    )
    report_json_path = Path(cfg_effective["paths"]["reports_dir"]) / (
        f"ptq_export_T{WINDOW_SIZE}_P{protocol}_{CALIBRATION_VARIANT}.json"
    )
    report_md_path = Path(cfg_effective["paths"]["reports_dir"]) / (
        f"ptq_export_T{WINDOW_SIZE}_P{protocol}_{CALIBRATION_VARIANT}.md"
    )

    export_json = {
        "window_size": int(WINDOW_SIZE),
        "protocol": protocol,
        "variant": CALIBRATION_VARIANT,
        "representative_samples": None,
        "tflite_model": None,
        "status": "error",
        "error": None,
    }
    eval_json = None

    try:
        rep_count = convert_to_int8_tflite(
            model=model,
            X_test=X_test,
            out_path=tflite_path,
            rep_samples=AUTHOR_STYLE_REP_SAMPLES,
        )
        eval_json = evaluate_int8_tflite_model(tflite_path, X_test, y_test)
        deployable_int8 = ("int8" in eval_json["input_dtype"].lower()) and ("int8" in eval_json["output_dtype"].lower())

        export_json.update(
            {
                "representative_samples": rep_count,
                "tflite_model": str(tflite_path),
                "model_size_kb": float(tflite_path.stat().st_size / 1024.0),
                "input_dtype": eval_json["input_dtype"],
                "output_dtype": eval_json["output_dtype"],
                "deployable_full_int8": bool(deployable_int8),
                "status": "ok",
                "error": None,
            }
        )
    except Exception as exc:
        export_json["error"] = str(exc)
        export_json["representative_samples"] = int(min(AUTHOR_STYLE_REP_SAMPLES, len(X_test)))
        if FAIL_FAST:
            raise

    dump_json(report_json_path, export_json)
    write_quant_report_md(
        report_md_path,
        f"PTQ Export (T={WINDOW_SIZE}, protocol={protocol}, variant={CALIBRATION_VARIANT})",
        export_json,
    )

    ptq_export_metrics[protocol] = export_json
    if eval_json is not None:
        ptq_eval_metrics[protocol] = eval_json

    ptq_rows.append(
        {
            "protocol": protocol,
            "variant": CALIBRATION_VARIANT,
            "representative_source": "test",
            "representative_samples": export_json.get("representative_samples"),
            "replication_metrics_status": "ok" if eval_json is not None else "error",
            "strict_deploy_status": "PASS" if export_json.get("deployable_full_int8") else "FAIL",
            "accuracy": eval_json.get("accuracy") if eval_json else None,
            "macro_f1": eval_json.get("macro_f1") if eval_json else None,
            "model_size_kb": eval_json.get("model_size_kb") if eval_json else export_json.get("model_size_kb"),
            "input_dtype": export_json.get("input_dtype"),
            "output_dtype": export_json.get("output_dtype"),
            "deployable_full_int8": export_json.get("deployable_full_int8", False),
            "status": export_json.get("status"),
            "failure_reason": export_json.get("error"),
            "tflite_model": export_json.get("tflite_model"),
        }
    )

ptq_df = pd.DataFrame(ptq_rows)
display(ptq_df)

INFO:tensorflow:Assets written to: /tmp/tmpnaf2c78r/assets


INFO:tensorflow:Assets written to: /tmp/tmpnaf2c78r/assets
/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-03-02 12:34:35.232009: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-02 12:34:35.232059: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-02 12:34:35.233474: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpnaf2c78r
2026-03-02 12:34:35.237217: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-02 12:34:35.237231: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpnaf2c78r
2026-03-02 12:34:35.248249: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 opt

INFO:tensorflow:Assets written to: /tmp/tmpfjdwgi4_/assets


INFO:tensorflow:Assets written to: /tmp/tmpfjdwgi4_/assets
/home/dellio/anaconda3/envs/tinymlproj/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-03-02 12:34:55.798010: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-02 12:34:55.798053: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-02 12:34:55.798466: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpfjdwgi4_
2026-03-02 12:34:55.804333: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-02 12:34:55.804350: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpfjdwgi4_
2026-03-02 12:34:55.824030: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
202

,protocol,variant,representative_source,representative_samples,replication_metrics_status,strict_deploy_status,accuracy,macro_f1,model_size_kb,input_dtype,output_dtype,deployable_full_int8,status,failure_reason,tflite_model
0,random_stratified,authorcal,test,100,ok,PASS,0.507755,0.436443,136.929688,<class 'numpy.int8'>,<class 'numpy.int8'>,True,ok,None,/home/dellio/github/har-mcu/models_tflite/deep...
1,user_holdout,authorcal,test,100,ok,PASS,0.125798,0.037247,136.929688,<class 'numpy.int8'>,<class 'numpy.int8'>,True,ok,None,/home/dellio/github/har-mcu/models_tflite/deep...


In [12]:
qat_export_metrics = {}
qat_eval_metrics = {}
qat_rows = []

if RUN_QAT:
    import tensorflow_model_optimization as tfmot

    for protocol in PROTOCOLS:
        arrays = load_split_arrays(cfg_effective["paths"]["processed_dir"], WINDOW_SIZE, protocol)
        X_train, y_train = arrays["X_train"], arrays["y_train"]
        X_val, y_val = arrays["X_val"], arrays["y_val"]
        X_test, y_test = arrays["X_test"], arrays["y_test"]

        tflite_path = Path(cfg_effective["paths"]["models_tflite_dir"]) / (
            f"deepconv_lstm_T{WINDOW_SIZE}_P{protocol}_qat_{CALIBRATION_VARIANT}.tflite"
        )
        report_json_path = Path(cfg_effective["paths"]["reports_dir"]) / (
            f"qat_export_T{WINDOW_SIZE}_P{protocol}_{CALIBRATION_VARIANT}.json"
        )
        report_md_path = Path(cfg_effective["paths"]["reports_dir"]) / (
            f"qat_export_T{WINDOW_SIZE}_P{protocol}_{CALIBRATION_VARIANT}.md"
        )

        export_json = {
            "window_size": int(WINDOW_SIZE),
            "protocol": protocol,
            "variant": CALIBRATION_VARIANT,
            "epochs_ran": 0,
            "representative_samples": int(min(AUTHOR_STYLE_REP_SAMPLES, len(X_test))),
            "tflite_model": None,
            "status": "error",
            "error": None,
        }
        eval_json = None

        try:
            num_classes = len(cfg_effective.get("classes", [])) or int(np.max(y_train)) + 1
            y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes)
            y_val_oh = tf.keras.utils.to_categorical(y_val, num_classes)

            ckpt_path = baseline_ckpt_path(cfg_effective["paths"]["checkpoints_dir"], WINDOW_SIZE, protocol)
            fp32_model = tf.keras.models.load_model(ckpt_path)

            try:
                qat_model = tfmot.quantization.keras.quantize_model(fp32_model)
            except Exception:
                quantize_annotate_layer = tfmot.quantization.keras.quantize_annotate_layer
                quantize_apply = tfmot.quantization.keras.quantize_apply

                def annotate(layer):
                    if isinstance(layer, (tf.keras.layers.Conv1D, tf.keras.layers.Dense)):
                        return quantize_annotate_layer(layer)
                    return layer

                annotated = tf.keras.models.clone_model(fp32_model, clone_function=annotate)
                annotated.set_weights(fp32_model.get_weights())
                qat_model = quantize_apply(annotated)

            qat_model.compile(
                optimizer=tf.keras.optimizers.RMSprop(
                    learning_rate=float(cfg_effective["quant"]["qat"].get("learning_rate", 1e-4))
                ),
                loss="categorical_crossentropy",
                metrics=["accuracy"],
            )

            hist = qat_model.fit(
                X_train,
                y_train_oh,
                validation_data=(X_val, y_val_oh),
                epochs=int(cfg_effective["quant"]["qat"].get("epochs", 10)),
                batch_size=int(cfg_effective["quant"]["qat"].get("batch_size", cfg_effective["train"].get("batch_size", 64))),
                verbose=2,
            )
            export_json["epochs_ran"] = int(len(hist.history.get("loss", [])))

            rep_count = convert_to_int8_tflite(
                model=qat_model,
                X_test=X_test,
                out_path=tflite_path,
                rep_samples=AUTHOR_STYLE_REP_SAMPLES,
            )
            eval_json = evaluate_int8_tflite_model(tflite_path, X_test, y_test)
            deployable_int8 = ("int8" in eval_json["input_dtype"].lower()) and ("int8" in eval_json["output_dtype"].lower())

            export_json.update(
                {
                    "representative_samples": rep_count,
                    "tflite_model": str(tflite_path),
                    "model_size_kb": float(tflite_path.stat().st_size / 1024.0),
                    "input_dtype": eval_json["input_dtype"],
                    "output_dtype": eval_json["output_dtype"],
                    "deployable_full_int8": bool(deployable_int8),
                    "status": "ok",
                    "error": None,
                }
            )
        except Exception as exc:
            export_json["error"] = str(exc)
            if FAIL_FAST:
                raise

        dump_json(report_json_path, export_json)
        write_quant_report_md(
            report_md_path,
            f"QAT Export (T={WINDOW_SIZE}, protocol={protocol}, variant={CALIBRATION_VARIANT})",
            export_json,
        )

        qat_export_metrics[protocol] = export_json
        if eval_json is not None:
            qat_eval_metrics[protocol] = eval_json

        qat_rows.append(
            {
                "protocol": protocol,
                "variant": CALIBRATION_VARIANT,
                "representative_source": "test",
                "representative_samples": export_json.get("representative_samples"),
                "replication_metrics_status": "ok" if eval_json is not None else "error",
                "strict_deploy_status": "PASS" if export_json.get("deployable_full_int8") else "FAIL",
                "accuracy": eval_json.get("accuracy") if eval_json else None,
                "macro_f1": eval_json.get("macro_f1") if eval_json else None,
                "model_size_kb": eval_json.get("model_size_kb") if eval_json else export_json.get("model_size_kb"),
                "input_dtype": export_json.get("input_dtype"),
                "output_dtype": export_json.get("output_dtype"),
                "deployable_full_int8": export_json.get("deployable_full_int8", False),
                "status": export_json.get("status"),
                "failure_reason": export_json.get("error"),
                "tflite_model": export_json.get("tflite_model"),
            }
        )
else:
    print("RUN_QAT=False -> skipping QAT section")

qat_df = pd.DataFrame(qat_rows)
display(qat_df)

,protocol,variant,representative_source,representative_samples,replication_metrics_status,strict_deploy_status,accuracy,macro_f1,model_size_kb,input_dtype,output_dtype,deployable_full_int8,status,failure_reason,tflite_model
0,random_stratified,authorcal,test,100,error,FAIL,None,None,None,None,None,False,error,Layer conv1:<class 'keras.src.layers.convoluti...,None
1,user_holdout,authorcal,test,100,error,FAIL,None,None,None,None,None,False,error,Layer conv1:<class 'keras.src.layers.convoluti...,None


In [13]:
TARGET_BASELINE_ACC = 0.9824
TARGET_PTQ_ACC = 0.9709
TARGET_PTQ_SIZE_KB = 136.51

primary_protocol = "random_stratified"

if primary_protocol not in baseline_eval_metrics:
    raise ValueError("random_stratified baseline results missing; cannot run verdict checks")

baseline_acc_random = float(baseline_eval_metrics[primary_protocol]["accuracy"])
ptq_export_primary = ptq_export_metrics.get(primary_protocol)
ptq_eval_primary = ptq_eval_metrics.get(primary_protocol)

ptq_replication_available = ptq_eval_primary is not None
strict_ptq_deployable = bool(
    ptq_export_primary
    and ptq_export_primary.get("status") == "ok"
    and ptq_export_primary.get("deployable_full_int8", False)
)

if ptq_replication_available:
    ptq_acc_random = float(ptq_eval_primary["accuracy"])
    ptq_size_random = float(ptq_eval_primary["model_size_kb"])
else:
    ptq_acc_random = None
    ptq_size_random = None

checks = [
    {
        "check": "Baseline accuracy close to paper target",
        "rule": "abs(baseline_acc - 0.9824) <= 0.02",
        "value": baseline_acc_random,
        "status": "PASS" if abs(baseline_acc_random - TARGET_BASELINE_ACC) <= 0.02 else "WARN",
    },
    {
        "check": "PTQ replication metrics available (authorcal)",
        "rule": "PTQ export + host TFLite eval completed",
        "value": "available" if ptq_replication_available else "missing",
        "status": "PASS" if ptq_replication_available else "FAIL",
    },
    {
        "check": "PTQ integer I/O deployment gate (authorcal)",
        "rule": "status=ok and input/output dtype are int8",
        "value": (
            f"status={ptq_export_primary.get('status') if ptq_export_primary else None}, "
            f"deployable={ptq_export_primary.get('deployable_full_int8') if ptq_export_primary else None}"
        ),
        "status": "PASS" if strict_ptq_deployable else "FAIL",
    },
]

if ptq_replication_available:
    checks.extend(
        [
            {
                "check": "PTQ accuracy close to paper target",
                "rule": "abs(ptq_acc - 0.9709) <= 0.03",
                "value": ptq_acc_random,
                "status": "PASS" if abs(ptq_acc_random - TARGET_PTQ_ACC) <= 0.03 else "WARN",
            },
            {
                "check": "PTQ size close to paper value",
                "rule": "abs(size_kb - 136.51) <= 60",
                "value": ptq_size_random,
                "status": "PASS" if abs(ptq_size_random - TARGET_PTQ_SIZE_KB) <= 60.0 else "WARN",
            },
        ]
    )

if RUN_QAT:
    qat_export_primary = qat_export_metrics.get(primary_protocol)
    qat_eval_primary = qat_eval_metrics.get(primary_protocol)
    qat_replication_available = qat_eval_primary is not None
    strict_qat_deployable = bool(
        qat_export_primary
        and qat_export_primary.get("status") == "ok"
        and qat_export_primary.get("deployable_full_int8", False)
)

    checks.extend(
        [
            {
                "check": "QAT replication metrics available (authorcal)",
                "rule": "QAT export + host TFLite eval completed",
                "value": "available" if qat_replication_available else "missing",
                "status": "PASS" if qat_replication_available else "FAIL",
            },
            {
                "check": "QAT integer I/O deployment gate (authorcal)",
                "rule": "status=ok and input/output dtype are int8",
                "value": (
                    f"status={qat_export_primary.get('status') if qat_export_primary else None}, "
                    f"deployable={qat_export_primary.get('deployable_full_int8') if qat_export_primary else None}"
                ),
                "status": "PASS" if strict_qat_deployable else "FAIL",
            },
        ]
    )

verdict_df = pd.DataFrame(checks)
display(verdict_df)


,check,rule,value,status
0,Baseline accuracy close to paper target,abs(baseline_acc - 0.9824) <= 0.02,0.511853,WARN
1,PTQ replication metrics available (authorcal),PTQ export + host TFLite eval completed,available,PASS
2,PTQ integer I/O deployment gate (authorcal),status=ok and input/output dtype are int8,"status=ok, deployable=True",PASS
3,PTQ accuracy close to paper target,abs(ptq_acc - 0.9709) <= 0.03,0.507755,WARN
4,PTQ size close to paper value,abs(size_kb - 136.51) <= 60,136.929688,PASS
5,QAT replication metrics available (authorcal),QAT export + host TFLite eval completed,missing,FAIL
6,QAT integer I/O deployment gate (authorcal),status=ok and input/output dtype are int8,"status=error, deployable=None",FAIL


In [14]:
summary_rows = []

for protocol in PROTOCOLS:
    if protocol in baseline_eval_metrics:
        m = baseline_eval_metrics[protocol]
        summary_rows.append(
            {
                "protocol": protocol,
                "model": "baseline",
                "variant": "n/a",
                "representative_source": "n/a",
                "accuracy": m["accuracy"],
                "macro_f1": m["macro_f1"],
                "model_size_kb": None,
                "replication_metrics_status": "ok",
                "strict_deploy_status": "n/a",
                "failure_reason": None,
            }
        )

    export_m = ptq_export_metrics.get(protocol)
    eval_m = ptq_eval_metrics.get(protocol)
    if export_m:
        strict_ok = bool(
            export_m.get("status") == "ok"
            and export_m.get("deployable_full_int8", False)
        )
        summary_rows.append(
            {
                "protocol": protocol,
                "model": "ptq",
                "variant": CALIBRATION_VARIANT,
                "representative_source": "test",
                "accuracy": eval_m.get("accuracy") if eval_m else None,
                "macro_f1": eval_m.get("macro_f1") if eval_m else None,
                "model_size_kb": (
                    eval_m.get("model_size_kb") if eval_m else export_m.get("model_size_kb")
                ),
                "replication_metrics_status": "ok" if eval_m else "error",
                "strict_deploy_status": "PASS" if strict_ok else "FAIL",
                "failure_reason": export_m.get("error"),
            }
        )

    if RUN_QAT:
        export_q = qat_export_metrics.get(protocol)
        eval_q = qat_eval_metrics.get(protocol)
        if export_q:
            strict_ok_q = bool(
                export_q.get("status") == "ok"
                and export_q.get("deployable_full_int8", False)
            )
            summary_rows.append(
                {
                    "protocol": protocol,
                    "model": "qat",
                    "variant": CALIBRATION_VARIANT,
                    "representative_source": "test",
                    "accuracy": eval_q.get("accuracy") if eval_q else None,
                    "macro_f1": eval_q.get("macro_f1") if eval_q else None,
                    "model_size_kb": eval_q.get("model_size_kb") if eval_q else export_q.get("model_size_kb"),
                    "replication_metrics_status": "ok" if eval_q else "error",
                    "strict_deploy_status": "PASS" if strict_ok_q else "FAIL",
                    "failure_reason": export_q.get("error"),
                }
            )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

reports_dir = Path(cfg_effective["paths"]["reports_dir"])
reports_dir.mkdir(parents=True, exist_ok=True)

summary_csv_path = reports_dir / f"notebook_replication_summary_T{WINDOW_SIZE}.csv"
summary_md_path = reports_dir / f"notebook_replication_summary_T{WINDOW_SIZE}.md"

summary_df.to_csv(summary_csv_path, index=False)

with summary_md_path.open("w", encoding="utf-8") as f:
    f.write(f"# Notebook Replication Summary (T={WINDOW_SIZE})\n\n")
    f.write(f"- Run mode: `{RUN_MODE}`\n")
    f.write(f"- Protocols: {PROTOCOLS}\n")
    f.write(f"- Calibration variant: `{CALIBRATION_VARIANT}` (test-based representative dataset)\n\n")
    f.write("## Summary table\n\n")
    f.write("```\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n```\n\n")
    f.write("## Verdict table\n\n")
    f.write("```\n")
    f.write(verdict_df.to_string(index=False))
    f.write("\n```\n")

print("Saved:")
print("-", summary_csv_path)
print("-", summary_md_path)

,protocol,model,variant,representative_source,accuracy,macro_f1,model_size_kb,replication_metrics_status,strict_deploy_status,failure_reason
0,random_stratified,baseline,n/a,n/a,0.511853,0.454694,NaN,ok,n/a,None
1,random_stratified,ptq,authorcal,test,0.507755,0.436443,136.929688,ok,PASS,None
2,random_stratified,qat,authorcal,test,NaN,NaN,NaN,error,FAIL,Layer conv1:<class 'keras.src.layers.convoluti...
3,user_holdout,baseline,n/a,n/a,0.070770,0.062999,NaN,ok,n/a,None
4,user_holdout,ptq,authorcal,test,0.125798,0.037247,136.929688,ok,PASS,None
5,user_holdout,qat,authorcal,test,NaN,NaN,NaN,error,FAIL,Layer conv1:<class 'keras.src.layers.convoluti...


Saved:
- /home/dellio/github/har-mcu/reports/notebook_replication_summary_T100.csv
- /home/dellio/github/har-mcu/reports/notebook_replication_summary_T100.md


In [15]:
repro_rows = []
DRIFT_TOL = 1e-9

for protocol in PROTOCOLS:
    split_path = split_npz_path(cfg_effective["paths"]["processed_dir"], WINDOW_SIZE, protocol)
    split_hash = None
    if split_path.exists():
        split_npz = np.load(split_path, allow_pickle=True)
        raw_hash = split_npz["split_hash"]
        split_hash = raw_hash.item() if hasattr(raw_hash, "item") else str(raw_hash)

    repeat_out = evaluate_baseline_for_protocol(cfg_effective, WINDOW_SIZE, protocol)
    repeat_metrics = load_json(repeat_out["metrics_json"])

    old_acc = float(baseline_eval_metrics[protocol]["accuracy"])
    new_acc = float(repeat_metrics["accuracy"])
    drift = abs(new_acc - old_acc)

    repro_rows.append(
        {
            "protocol": protocol,
            "split_hash": split_hash,
            "baseline_acc_first": old_acc,
            "baseline_acc_repeat": new_acc,
            "abs_drift": drift,
            "status": "PASS" if drift <= DRIFT_TOL else "WARN",
        }
    )

repro_df = pd.DataFrame(repro_rows)
display(repro_df)


,protocol,split_hash,baseline_acc_first,baseline_acc_repeat,abs_drift,status
0,random_stratified,e8b3eefc294da8de,0.511853,0.511853,0.0,PASS
1,user_holdout,8f23a2e13f210a67,0.070770,0.070770,0.0,PASS


## Interpretation Notes and Limitations

- This notebook remains **methodologically safe**: train-only normalization, per-user-safe window construction, and a dedicated validation split for training-time callbacks.
- PTQ and QAT export logic is intentionally simplified to a direct converter flow modeled after v2 PTQ, but applied on top of the safer v1 dataset/training pipeline.
- Calibration variant in this notebook is unified to `authorcal` only (representative samples from test split) for straightforward comparison across PTQ and QAT.
- `replication_metrics_status` reflects host-side TFLite evaluation availability.
- `strict_deploy_status` in this notebook is an integer I/O gate (`status=ok` and int8 input/output dtypes); it is not a full TFLM op-compatibility audit.

## Suggested next steps
- Run `RUN_MODE = "full"` for final replication logs.
- Compare baseline vs PTQ vs QAT on each protocol from `summary_df`.
- If needed for deployment readiness, run the stricter script-level export gates in `src/quant/` after notebook prototyping.
